In [1]:
import pandas as pd

# Extract

In [6]:

# Energy data
energy_data = pd.read_excel('Generated_Data_Model.xlsx', sheet_name='Energy_data')
# Weather data
weather_data = pd.read_excel('Generated_Data_Model.xlsx', sheet_name='Weather_data_per_zone')
# Sun data
sun_data = pd.read_excel('Generated_Data_Model.xlsx', sheet_name='Sun_data')

In [ ]:
energy_data["Timestamp"] = pd.to_datetime(energy_data["Timestamp"])
weather_data["Timestamp"] = pd.to_datetime(weather_data["Timestamp"])
sun_data["Timestamp"] = pd.to_datetime(sun_data["Timestamp"])

expected_columns_energy_train = ['Timestamp','Price']
expected_columns_energy_test = ['Timestamp']
expected_columns_weather = ["MaxTemp", "MinTemp", "UvIndex", "Wind", "Dew point", "Cloud cover", "Wind_speed", "Pressure", "Apparent_temp_max", "Apparent_temp_min", "Visibility", "Humidity"]
sun_data_columns = ["Sunrise", "Sunset","Solar_noon", "Day_length"]

# Save expected columns to a .txt file
with open('expected_columns.txt', 'w') as f:
    f.write("Expected columns for energy_train:\n")
    f.write(", ".join(expected_columns_energy_train) + "\n\n")
    
    f.write("Expected columns for energy_test:\n")
    f.write(", ".join(expected_columns_energy_test) + "\n\n")
    
    f.write("Expected columns for weather_data:\n")
    f.write(", ".join(expected_columns_weather) + "\n\n")
    
    f.write("Expected columns for sun_data:\n")
    f.write(", ".join(sun_data_columns) + "\n")

with open('expected_columns.txt', 'r') as f:
    expected_columns_energy_train = f.read().split("Expected columns for energy_train:\n")[1].split("\n\n")[0].split(", ")
    expected_columns_energy_test = f.read().split("Expected columns for energy_test:\n")[1].split("\n\n")[0].split(", ")
    expected_columns_weather = f.read().split("Expected columns for weather_data:\n")[1].split("\n\n")[0].split(", ")
    sun_data_columns = f.read().split("Expected columns for sun_data:\n")[1].split("\n\n")[0].split(", ")

def check_columns(df, expected_columns):
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        print(f"Missing columns: {missing_columns}")
    else:
        print("All expected columns are present.")
        

In [ ]:
# Time data
def generate_time_data(start_date, end_date, country_holidays):
    time_data = pd.DataFrame({'Timestamp': pd.date_range(start=start_date, end=end_date, freq='H')})
    time_data['Hour'] = time_data['Timestamp'].dt.hour
    time_data['Day'] = time_data['Timestamp'].dt.day
    time_data['Month'] = time_data['Timestamp'].dt.month
    time_data['Year'] = time_data['Timestamp'].dt.year
    time_data['DayOfWeek'] = time_data['Timestamp'].dt.dayofweek
    time_data['DayOfYear'] = time_data['Timestamp'].dt.dayofyear
    time_data['IsWeekend'] = time_data['DayOfWeek'].isin([5, 6]).astype(bool)
    # Get the holidays for the country
    import holidays
    country_holidays = holidays.CountryHoliday(country_holidays)
    time_data['IsHoliday'] = time_data['Timestamp'].dt.date.astype('datetime64').isin(country_holidays).astype(bool)
    working_days = (time_data['IsWeekend'] == 0) & (time_data['IsHoliday'] == 0)
    time_data['IsWorkingDay'] = working_days.astype(bool)
    return time_data

# Transform

In [ ]:
import numpy as np
def transform_time_data(time_data):
    # All bool remain as bool, no transformation needed
    # All other columns sin cos transformation
    for col in time_data.columns:
        if time_data[col].dtype == 'bool':
            continue
        else:
            max_val = time_data[col].max()
            time_data[f'{col}_sin'] = np.sin(2 * np.pi * time_data[col] / max_val)
            time_data[f'{col}_cos'] = np.cos(2 * np.pi * time_data[col] / max_val)
            time_data.drop(columns=[col], inplace=True)
    
    return time_data

def transform_weather_data(weather_data):
    # Normalize all columns except Timestamp
    for col in weather_data.columns:
        if col == 'Timestamp':
            continue
        else:
            weather_data[col] = (weather_data[col] - weather_data[col].min()) / (weather_data[col].max() - weather_data[col].min())
    
    return weather_data

def transform_sun_data(sun_data):
    # Transform to hourly data by forward filling the values for each hour
    sun_data = sun_data.set_index('Timestamp').resample('H').ffill().reset_index()
    sun_data["Solar_intensity"] = max(0,np.cos((sun_data['Timestamp'] -sun_data['Solar_noon']) * np.pi / sun_data['Day_length']))
    sun_data.drop(columns=['Sunrise', 'Sunset', 'Solar_noon', 'Day_length'], inplace=True)
    
    
    return sun_data

def merge_datasets(energy_data, weather_data, sun_data, time_data):
    # Merge on Timestamp
    merged_data = energy_data.merge(weather_data, on='Timestamp', how='left')
    merged_data = merged_data.merge(sun_data, on='Timestamp', how='left')
    merged_data = merged_data.merge(time_data, on='Timestamp', how='left')
    
    return merged_data





    


,Timestamp,Zone,Energy compsuption,Price
0,2024-01-01 00:00:00,Zone A,71.882474,0.154721
1,2024-01-01 01:00:00,Zone A,63.171206,0.137001
2,2024-01-01 02:00:00,Zone A,65.097858,0.152684
3,2024-01-01 03:00:00,Zone A,63.679650,0.163671
4,2024-01-01 04:00:00,Zone A,65.924442,0.182528
...,...,...,...,...
17563,2024-12-31 19:00:00,Zone B,74.558294,0.186101
17564,2024-12-31 20:00:00,Zone B,81.938696,0.173015
17565,2024-12-31 21:00:00,Zone B,75.216683,0.174675
17566,2024-12-31 22:00:00,Zone B,62.555159,0.159031
